# Titans memory adapter v2 ? fact-mix retrieval

B?n train n?y th?c hi?n Phase 0 cho **m?t c?u h?nh Titans**: Qwen2.5-0.5B ??ng b?ng, Titans NeuralMemory c? `init_decay_bias=-5`, QK RMSNorm, v? objective tr?n next-token Wikipedia v?i c?c truy v?n s? ki?n b?a c? tr?ng s? ??p ?n 8?.

Train t? ??u, seed c? ??nh. T?n/template train, validation v? test t?ch ri?ng. Notebook ch? ??nh gi? Titans tr?n t?p s? ki?n gi? l?i; ch?a so s?nh v?i TTT. K?t qu? ?m v?n ???c gi? nguy?n, kh?ng m? retrieval gate.

C?n Kaggle GPU, Internet, v? dataset input `thanh70kg/titans-vs-ttt-input` c? `data_vi.pt`. Output: `/kaggle/working/titans_v2.pt`, `/kaggle/working/eval_v2.json`, log.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "titans-pytorch==0.6.0", "transformers", "datasets", "accelerate", "matplotlib"])

In [ ]:
import os, time, math, json, re, random, glob, html as _html
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

DEBUG = False   # True = chạy thử nhanh bằng model ngẫu nhiên nhỏ, không cần GPU/Internet
MEMORY_KIND = "titans"    # "titans" | "ttt_linear" | "ttt_mlp"   (đổi dòng này để train biến thể khác)

# ---- Backbone & dữ liệu (GIỮ NGUYÊN giữa các notebook để so sánh công bằng) ----
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
SEG_LEN    = 256      # độ dài 1 segment = cửa sổ mà backbone nhìn thấy
N_SEGS     = 9        # số segment liên tiếp của 1 tài liệu khi train
TBPTT      = 2        # backward sau mỗi TBPTT segment (cắt gradient qua state memory)
N_TRAIN_DOCS = 6000
N_EVAL_DOCS  = 24
# Loại các bài trùng chủ đề vật lý khỏi dữ liệu train (tránh rò rỉ khi test bằng sách vật lý)
EXCLUDE_TITLE_REGEX = r"vật lý|cơ học|nhiệt động|điện từ|lượng tử|tương đối|newton|động lực học|năng lượng"

# ---- Memory (cùng kích thước cho cả hai: 4 head x 64 = 256 chiều, block/chunk = 16 như paper TTT) ----
MEM_CHUNK, MEM_HEADS, MEM_DIM_HEAD, MEM_DEPTH = 16, 4, 64, 2
TITANS_INNER_LR_CAP = 1e-2
TITANS_KWARGS = dict(init_decay_bias=-5.0, init_momentum_bias=2.0,
                     init_adaptive_step_bias=0.0, qk_rmsnorm=True, max_grad_norm=1.0)
FACT_ANSWER_WEIGHT = 8.0
FACTMIX_DISTANCES = [0, 1, 2, 4, 7]
USE_MOMENTUM = True     # chỉ dùng cho Titans; đặt False để chạy ablation (train lại) không có momentum
HOOK_LAYER   = None     # None = giữa mạng

# ---- Train ----
BATCH_DOCS, LR, WARMUP, MAX_STEPS = 2, 1e-3, 100, 3000
EVAL_EVERY, SAVE_EVERY, MAX_MINUTES = 200, 100, 300
CKPT_DIR  = os.environ.get("CKPT_DIR", f"/kaggle/working/ckpt_{MEMORY_KIND}")
DATA_PATH = os.environ.get("DATA_PATH", "/kaggle/input/titans-vs-ttt-input/data_vi.pt")
INIT_CKPT = None  # v2 train từ đầu; không resume checkpoint v1
SEED = 42

if DEBUG:
    SEG_LEN, N_SEGS, TBPTT = 32, 4, 2
    MEM_CHUNK, MEM_HEADS, MEM_DIM_HEAD = 4, 2, 16
    N_TRAIN_DOCS, N_EVAL_DOCS, MAX_STEPS = 16, 4, 6
    WARMUP, EVAL_EVERY, SAVE_EVERY = 2, 3, 3

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| DEBUG:", DEBUG, "| memory:", MEMORY_KIND)
try:
    from IPython.display import HTML, display
except Exception:
    HTML = lambda s: s
    display = print

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Qwen2Config, Qwen2ForCausalLM

if DEBUG:
    torch.manual_seed(1234)     # backbone ngẫu nhiên phải giống nhau giữa các notebook DEBUG
    _cfg = Qwen2Config(vocab_size=500, hidden_size=64, intermediate_size=128, num_hidden_layers=4,
                       num_attention_heads=4, num_key_value_heads=2, max_position_embeddings=4096)
    lm = Qwen2ForCausalLM(_cfg)
    def encode(t): return [b % 500 for b in t.encode("utf-8")]
    def decode(ids): return "".join(chr(32 + (int(i) % 90)) for i in ids)
else:
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    _dt = torch.float32  # Avoid FP16 overflow in the frozen backbone on T4
    try:
        lm = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=_dt)
    except TypeError:
        lm = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=_dt)
    def encode(t): return tok(t, add_special_tokens=False)["input_ids"]
    def decode(ids): return tok.decode(ids)

lm = lm.to(device).eval()
for p in lm.parameters():
    p.requires_grad_(False)
print("tham số backbone (đóng băng), triệu:", round(sum(p.numel() for p in lm.parameters()) / 1e6, 1))

## 1. Dữ liệu pretrain cho memory: Wikipedia tiếng Việt

Tài liệu dài, cắt đúng `SEG_LEN * N_SEGS` token. Các bài có tiêu đề trùng chủ đề vật lý bị loại (xem `EXCLUDE_TITLE_REGEX`). Mười mấy bài cuối làm tập đánh giá, **không train**.

In [ ]:
def build_dataset():
    from datasets import load_dataset
    need = SEG_LEN * N_SEGS
    ds = load_dataset("wikimedia/wikipedia", "20231101.vi", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=10000)
    rx = re.compile(EXCLUDE_TITLE_REGEX, re.I)
    out, seen = [], 0
    for ex in ds:
        seen += 1
        if rx.search(ex["title"]) or len(ex["text"]) < need * 2:
            continue
        ids = encode(ex["text"])
        if len(ids) < need:
            continue
        out.append(torch.tensor(ids[:need]))
        if len(out) % 500 == 0:
            print(len(out), "tài liệu (đã duyệt", seen, "bài)")
        if len(out) >= N_TRAIN_DOCS + N_EVAL_DOCS:
            break
    return torch.stack(out)

_found = [DATA_PATH] if os.path.exists(DATA_PATH) else sorted(glob.glob("/kaggle/input/**/data_vi.pt", recursive=True))
if DEBUG:
    data = torch.randint(0, lm.config.vocab_size, (N_TRAIN_DOCS + N_EVAL_DOCS, SEG_LEN * N_SEGS))
elif _found:
    print("Dùng dữ liệu đã lưu:", _found[0])
    data = torch.load(_found[0])
else:
    data = build_dataset()
    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
    torch.save(data, DATA_PATH)

# Dataset cache v1 c? 8 segment; v2 th?m m?t segment ?? ch?a query sau 7 segment nhi?u.
if data.ndim != 2:
    raise ValueError(f"data_vi.pt ph?i l? tensor 2 chi?u, nh?n {tuple(data.shape)}")
if data.size(1) == SEG_LEN * (N_SEGS - 1):
    data = torch.cat([data, data[:, :SEG_LEN]], dim=1)
elif data.size(1) != SEG_LEN * N_SEGS:
    raise ValueError(f"Dataset ph?i c? {SEG_LEN*(N_SEGS-1)} ho?c {SEG_LEN*N_SEGS} token/doc; nh?n {data.size(1)}")

train_docs, eval_docs = data[:-N_EVAL_DOCS], data[-N_EVAL_DOCS:]
print("train:", tuple(train_docs.shape), "| eval:", tuple(eval_docs.shape),
      "| tổng token train (triệu):", round(train_docs.numel() / 1e6, 1))

## 2. Mô hình: backbone đóng băng + Neural Memory kiểu Titans

Memory được gắn bằng *forward hook* vào đầu ra của một layer giữa mạng. Ở mỗi segment: hidden state của segment được **ghi** vào memory (cập nhật fast-weight bằng gradient của `‖M(k)−v‖²` có momentum và weight decay/forgetting), rồi memory **đọc** lại và cộng vào hidden state qua `out` và `gate`. Việc đọc dùng trọng số memory của các chunk *trước* nên đúng tính nhân quả. Backbone mỗi lần chỉ thấy `SEG_LEN` token; mọi thông tin xa hơn phải đi qua memory.

In [ ]:
# ---- TTT layer (đã kiểm tra số học: dual form == vòng lặp từng token, sai số ~1e-15) ----
# ==== TTT layer (Sun et al., "Learning to (Learn at Test Time)") dùng làm memory phụ ====
# Cài đặt riêng theo mô tả trong paper (không phải code chính thức):
#   loss      l(W; x) = 1/2 || f(theta_K x; W) - theta_V x ||^2           (Eq. 4, hệ số 1/2 gộp vào lr)
#   f(x; W)   = x + LN(f_res(x; W)),  f_res = Linear (TTT-Linear) hoac MLP 2 lop GELU, hidden = 4 * dim (TTT-MLP)
#   mini-batch TTT (b = block): gradient cua moi token trong block tinh tai W dau block (t' = t - mod(t, b));
#       W_t = W_t' - sum_{s=t'+1..t} eta_s * grad l(W_t'; x_s)             (mini-batch GD)
#   output    z_t = f(theta_Q x_t; W_t)                                    (Eq. 5)
#   eta(x)    = eta_base * sigmoid(theta_lr . x)  (eta_base = 1 Linear, 0.1 MLP);  W_0 hoc duoc (outer loop)
# Tinh toan bang "dual form": khong tao W_t rieng cho tung token, ket qua tuong duong ve mat toan hoc.
# Ghi chu: chia eta cho head_dim la lua chon de on dinh (theo thong le cua cac ban cai dat tham khao), khong co trong paper.
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


def ln_fwd(x, gamma, beta, eps=1e-6):
    mu = x.mean(-1, keepdim=True)
    var = x.var(-1, unbiased=False, keepdim=True)
    return gamma * (x - mu) / torch.sqrt(var + eps) + beta


def ln_l2_bwd(x, target, gamma, beta, eps=1e-6):
    # gradient theo x cua 1/2 || LN(x) - target ||^2 (dang dong kin)
    D = x.shape[-1]
    mu = x.mean(-1, keepdim=True)
    var = x.var(-1, unbiased=False, keepdim=True)
    std = torch.sqrt(var + eps)
    x_hat = (x - mu) / std
    y = gamma * x_hat + beta
    g = (y - target) * gamma
    return (D * g - g.sum(-1, keepdim=True) - x_hat * (g * x_hat).sum(-1, keepdim=True)) / (D * std)


def gelu_grad(x):
    return 0.5 * (1 + torch.erf(x / math.sqrt(2))) + x * torch.exp(-0.5 * x * x) / math.sqrt(2 * math.pi)


def _lower_eta(eta, k):
    # eta: (B, nh, k) -> E[t, s] = eta_s voi s <= t
    return torch.tril(eta.unsqueeze(-2).expand(*eta.shape[:-1], k, k))


def ttt_linear_block(st, q, k, v, eta, ln_w, ln_b):
    # q, k, v: (B, nh, kk, f); eta: (B, nh, kk); st: W1 (B,nh,f,f), b1 (B,nh,1,f)
    W, b = st["W1"], st["b1"]
    kk = k.size(-2)
    Z1 = k @ W + b
    G = ln_l2_bwd(Z1, v - k, ln_w, ln_b)
    loss_tok = ((k + ln_fwd(Z1, ln_w, ln_b) - v) ** 2).mean(-1)          # (B, nh, kk) "surprise"
    E = _lower_eta(eta, kk)
    A = torch.tril(q @ k.transpose(-1, -2))
    Zb = q @ W - (E * A) @ G + (b - E @ G)
    out = q + ln_fwd(Zb, ln_w, ln_b)
    W_new = W - (eta.unsqueeze(-1) * k).transpose(-1, -2) @ G
    b_new = b - (eta.unsqueeze(-1) * G).sum(-2, keepdim=True)
    return out, {"W1": W_new, "b1": b_new}, loss_tok


def ttt_mlp_block(st, q, k, v, eta, ln_w, ln_b):
    W1, b1, W2, b2 = st["W1"], st["b1"], st["W2"], st["b2"]
    kk = k.size(-2)
    Z1 = k @ W1 + b1
    X2 = F.gelu(Z1)
    Z2 = X2 @ W2 + b2
    G2 = ln_l2_bwd(Z2, v - k, ln_w, ln_b)
    G1 = (G2 @ W2.transpose(-1, -2)) * gelu_grad(Z1)
    loss_tok = ((k + ln_fwd(Z2, ln_w, ln_b) - v) ** 2).mean(-1)
    E = _lower_eta(eta, kk)
    A1 = torch.tril(q @ k.transpose(-1, -2))
    Z1b = q @ W1 - (E * A1) @ G1 + (b1 - E @ G1)
    X2b = F.gelu(Z1b)
    A2 = torch.tril(X2b @ X2.transpose(-1, -2))
    Z2b = X2b @ W2 - (E * A2) @ G2 + (b2 - E @ G2)
    out = q + ln_fwd(Z2b, ln_w, ln_b)
    e = eta.unsqueeze(-1)
    new = {
        "W1": W1 - (e * k).transpose(-1, -2) @ G1, "b1": b1 - (e * G1).sum(-2, keepdim=True),
        "W2": W2 - (e * X2).transpose(-1, -2) @ G2, "b2": b2 - (e * G2).sum(-2, keepdim=True),
    }
    return out, new, loss_tok


def flat_state(state):
    return torch.cat([state[n].flatten(1) for n in sorted(state)], dim=-1)


class TTTMem(nn.Module):
    """Giao dien: forward(x:(B,T,dim), state, want_trace) -> (read:(B,T,dim), new_state, info)."""

    def __init__(self, dim, heads=4, head_dim=64, block=16, kind="linear", eta_base=None):
        super().__init__()
        assert kind in ("linear", "mlp")
        self.nh, self.f, self.b, self.kind = heads, head_dim, block, kind
        self.eta_base = eta_base if eta_base is not None else (1.0 if kind == "linear" else 0.1)
        inner = heads * head_dim
        self.norm = RMSNorm(dim)
        self.to_q = nn.Linear(dim, inner, bias=False)
        self.to_k = nn.Linear(dim, inner, bias=False)
        self.to_v = nn.Linear(dim, inner, bias=False)
        self.to_lr = nn.Linear(dim, heads)
        nn.init.normal_(self.to_lr.weight, std=0.02)
        nn.init.zeros_(self.to_lr.bias)
        self.merge = nn.Linear(inner, dim, bias=False)
        self.ln_w = nn.Parameter(torch.ones(heads, 1, head_dim))
        self.ln_b = nn.Parameter(torch.zeros(heads, 1, head_dim))
        f = head_dim
        if kind == "linear":
            self.W1 = nn.Parameter(torch.randn(heads, f, f) * 0.02)
            self.b1 = nn.Parameter(torch.zeros(heads, 1, f))
            self.state_names = ["W1", "b1"]
        else:
            h = 4 * f
            self.W1 = nn.Parameter(torch.randn(heads, f, h) * 0.02)
            self.b1 = nn.Parameter(torch.zeros(heads, 1, h))
            self.W2 = nn.Parameter(torch.randn(heads, h, f) * 0.02)
            self.b2 = nn.Parameter(torch.zeros(heads, 1, f))
            self.state_names = ["W1", "b1", "W2", "b2"]

    def init_state(self, B):
        return {n: getattr(self, n).unsqueeze(0).expand(B, *getattr(self, n).shape).clone() for n in self.state_names}

    def forward(self, x, state=None, want_trace=False):
        B, T, _ = x.shape
        h = self.norm(x)
        sp = lambda t: t.view(B, T, self.nh, self.f).transpose(1, 2)
        q, k, v = sp(self.to_q(h)), sp(self.to_k(h)), sp(self.to_v(h))
        eta = self.eta_base * torch.sigmoid(self.to_lr(h)).transpose(1, 2) / self.f      # (B, nh, T)
        if state is None:
            state = self.init_state(B)
        block = ttt_linear_block if self.kind == "linear" else ttt_mlp_block
        outs, losses = [], []
        snaps = [flat_state(state).detach()] if want_trace else None
        for s in range(0, T, self.b):
            e = min(s + self.b, T)
            o, state, l = block(state, q[:, :, s:e], k[:, :, s:e], v[:, :, s:e], eta[:, :, s:e], self.ln_w, self.ln_b)
            outs.append(o)
            losses.append(l)
            if want_trace:
                snaps.append(flat_state(state).detach())
        z = torch.cat(outs, 2).transpose(1, 2).reshape(B, T, -1)
        read = self.merge(z)
        info = None
        if want_trace:
            info = dict(surprise=torch.cat(losses, 2).detach().float().mean((0, 1)),
                        lr=eta.detach().float().mean((0, 1)),
                        snaps=torch.stack(snaps, dim=1))
        return read, state, info

In [ ]:
from titans_pytorch import NeuralMemory, mem_state_detach

class MemLM(nn.Module):
    # Backbone đóng băng + một module memory gắn vào hidden state ở giữa mạng.
    # Cùng cách đọc ra (out, gate) cho mọi loại memory để so sánh công bằng.
    def __init__(self, lm, kind, layer=None):
        super().__init__()
        self.lm, self.kind = lm, kind
        d = lm.config.hidden_size
        if kind == "titans":
            # batch_size=MEM_CHUNK: sau mỗi chunk memory mới được cập nhật và chunk sau tính gradient theo trọng số MỚI
            # (mini-batch gradient descent của paper). Nếu để batch_size=None, thư viện luôn tính surprise theo trọng số
            # khởi tạo nên memory không bao giờ ảnh hưởng tới surprise.
            self.mem = NeuralMemory(dim=d, chunk_size=MEM_CHUNK, batch_size=MEM_CHUNK, heads=MEM_HEADS, dim_head=MEM_DIM_HEAD,
                                    momentum=USE_MOMENTUM, default_step_transform_max_lr=TITANS_INNER_LR_CAP,
                                    default_model_kwargs=dict(depth=MEM_DEPTH, expansion_factor=4.0), **TITANS_KWARGS)
            # titans-pytorch tạo tham số khởi tạo memory theo head bằng `repeat` (view stride 0) -> Adam không ghi được.
            for i, p in enumerate(self.mem.memory_model_parameters):
                self.mem.memory_model_parameters[i] = nn.Parameter(p.detach().clone().contiguous())
        elif kind in ("ttt_linear", "ttt_mlp"):
            self.mem = TTTMem(d, MEM_HEADS, MEM_DIM_HEAD, MEM_CHUNK, "linear" if kind == "ttt_linear" else "mlp")
        else:
            raise ValueError(kind)
        self.out = nn.Linear(d, d, bias=False)
        nn.init.normal_(self.out.weight, std=0.02)
        self.gate = nn.Parameter(torch.tensor(0.5))
        self.layer = layer if layer is not None else lm.config.num_hidden_layers // 2
        self.state, self.enabled, self.trace = None, True, False
        self.calls, self.w_ref = [], None
        lm.model.layers[self.layer].register_forward_hook(self._hook)

    def reset(self):
        self.state = None

    def detach_state(self):
        if self.state is None:
            return
        if self.kind == "titans":
            self.state = mem_state_detach(self.state)
        else:
            self.state = {k: v.detach() for k, v in self.state.items()}

    def fast_params(self):
        if self.kind == "titans":
            return sum(v.numel() for k, v in self.state.weights.items() if k.startswith("model.weights"))
        return sum(v.numel() for v in self.state.values())

    def trainable_parameters(self):
        return list(self.mem.parameters()) + list(self.out.parameters()) + [self.gate]

    def _mem_call(self, h):
        if self.kind == "titans":
            r, st, sur = self.mem(h, state=self.state, return_surprises=True)
            info = None
            if self.trace:
                flat = torch.cat([v.flatten(2) for k, v in st.updates.items() if k.startswith("model.weights")], dim=-1)
                info = dict(surprise=sur[0].detach().float().mean((0, 1)), lr=sur[1].detach().float().mean((0, 1)),
                            snaps=flat.detach().float())
            return r, st, info
        return self.mem(h, state=self.state, want_trace=self.trace)

    def _hook(self, module, args, output):
        if not self.enabled:
            return output
        h = output[0] if isinstance(output, tuple) else output
        r, st, info = self._mem_call(h.float())
        self.state = st
        if self.trace and info is not None:
            with torch.no_grad():
                snaps = info["snaps"]                                   # (B', n+1, P), snaps[:, 0] = trạng thái trước lần gọi
                if self.w_ref is None:
                    self.w_ref = snaps[:, 0].clone()
                dabs = (snaps - self.w_ref[:, None]).norm(dim=-1)
                dist = (dabs / self.w_ref.norm(dim=-1, keepdim=True)).mean(0)
                step = (snaps[:, 1:] - snaps[:, :-1]).norm(dim=-1).mean(0)
                self.calls.append(dict(surprise=info["surprise"].cpu().numpy(), theta=info["lr"].cpu().numpy(),
                                       dist=dist.cpu().numpy(), dist_abs=dabs.mean(0).cpu().numpy(), step=step.cpu().numpy()))
        h2 = h + (self.gate * self.out(r)).to(h.dtype)
        return (h2,) + tuple(output[1:]) if isinstance(output, tuple) else h2

def build_model(kind):
    m = MemLM(lm, kind, HOOK_LAYER)
    m.mem.to(device); m.out.to(device); m.gate.data = m.gate.data.to(device)
    return m

In [ ]:
model = build_model(MEMORY_KIND)
n_train = sum(p.numel() for p in model.trainable_parameters())
print(f"memory: {MEMORY_KIND} | hook ở layer {model.layer}/{lm.config.num_hidden_layers} | tham số được train (triệu): {n_train/1e6:.2f}")

## 3. Hàm loss, đánh giá và checkpoint

In [ ]:
TRAIN_NAMES = ["Zorvan", "Kelmar", "Tavris", "Brunex", "Ondira", "Veltro"]
TRAIN_METALS = ["Vexa", "Lorin", "Quasa", "Dermik", "Faltor"]

# Validation names/metals and wording are disjoint from both the training and test sets.
VAL_FACTS = [
    {"name":"Avenor", "metal":"Silex", "value":"4,18"},
    {"name":"Brelka", "metal":"Tormin", "value":"8,42"},
    {"name":"Cindrel", "metal":"Ulvax", "value":"2,67"},
    {"name":"Dovira", "metal":"Praxen", "value":"6,35"},
]
TEST_FACTS = [
    {"name":"Elarin", "metal":"Quorite", "value":"1,26"},
    {"name":"Fensal", "metal":"Mordane", "value":"3,84"},
    {"name":"Gavrik", "metal":"Nexium", "value":"5,19"},
    {"name":"Helvra", "metal":"Orralite", "value":"7,53"},
    {"name":"Istren", "metal":"Peldor", "value":"9,28"},
    {"name":"Jorvik", "metal":"Ruvane", "value":"2,91"},
    {"name":"Kelmor", "metal":"Sarnite", "value":"4,63"},
    {"name":"Luneth", "metal":"Tavrite", "value":"6,07"},
    {"name":"Mavren", "metal":"Uldane", "value":"8,34"},
    {"name":"Norell", "metal":"Vaskor", "value":"1,79"},
    {"name":"Orvane", "metal":"Wexite", "value":"3,52"},
    {"name":"Praxil", "metal":"Yondrel", "value":"5,96"},
]
FACT_SPLIT_META = {"train_names":TRAIN_NAMES, "train_metals":TRAIN_METALS,
                   "validation":VAL_FACTS, "test":TEST_FACTS,
                   "distances_segments":FACTMIX_DISTANCES, "answer_weight":FACT_ANSWER_WEIGHT,
                   "seed":SEED, "training_template":"S? th?c nghi?m ghi nh?n: m?u {metal} t?i tr?m {name} c? ch? s? {value} ??n v?.",
                   "eval_template":"Phi?u l?u tr? x?c nh?n ch? s? c?a h?p kim {metal} t?i tr?m {name} l? {value} ??n v?."}

def inject_factmix(batch_ids, step):
    """Replace a short tail in two chunks: a fact, then a delayed answer query."""
    ids = batch_ids.clone()
    weights = torch.ones_like(ids, dtype=torch.float32)
    rng = random.Random(SEED * 1000003 + step)
    for b in range(ids.size(0)):
        distance = rng.choice(FACTMIX_DISTANCES)
        event_seg = rng.randint(0, N_SEGS - distance - 2)
        query_seg = event_seg + distance + 1
        name, metal = rng.choice(TRAIN_NAMES), rng.choice(TRAIN_METALS)
        value = f"{rng.randint(1,9)},{rng.randint(10,99):02d}"
        fact = f"S? th?c nghi?m ghi nh?n: m?u {metal} t?i tr?m {name} c? ch? s? {value} ??n v?."
        prompt = f"Tr? l?i theo s? th?c nghi?m, m?u {metal} t?i tr?m {name} c? ch? s?"
        answer = f" {value}"
        fact_ids = encode(fact)
        prompt_ids, answer_ids = encode(prompt), encode(answer)
        query_ids = prompt_ids + answer_ids
        if len(fact_ids) >= SEG_LEN or len(query_ids) >= SEG_LEN:
            raise ValueError("Template fact/query d?i qu? m?t segment")
        fact_end = (event_seg + 1) * SEG_LEN
        ids[b, fact_end-len(fact_ids):fact_end] = torch.tensor(fact_ids, dtype=ids.dtype)
        query_end = (query_seg + 1) * SEG_LEN
        query_start = query_end - len(query_ids)
        ids[b, query_start:query_end] = torch.tensor(query_ids, dtype=ids.dtype)
        answer_start = query_start + len(prompt_ids)
        weights[b, answer_start:answer_start+len(answer_ids)] = FACT_ANSWER_WEIGHT
    return ids, weights

def doc_losses(model, ids, train=False, scaler=None, reset_each=False, token_weights=None):
    # ids: (B, N_SEGS*SEG_LEN). Trả về list loss trung bình từng segment.
    per_seg = []
    model.reset()
    for g in range(0, N_SEGS, TBPTT):
        group_loss = 0.0
        for s in range(g, min(g + TBPTT, N_SEGS)):
            if reset_each:
                model.reset()
            seg = ids[:, s * SEG_LEN:(s + 1) * SEG_LEN]
            logits = model.lm(input_ids=seg, use_cache=False).logits
            token_loss = F.cross_entropy(logits[:, :-1].float().reshape(-1, logits.size(-1)),
                                         seg[:, 1:].reshape(-1), reduction="none").reshape(seg.size(0), -1)
            if token_weights is None:
                l = token_loss.mean()
            else:
                target_weights = token_weights[:, s * SEG_LEN + 1:(s + 1) * SEG_LEN]
                l = (token_loss * target_weights).mean()
            loss_value = l.item()
            if not math.isfinite(loss_value):
                raise FloatingPointError(f"Non-finite loss at segment {s}: {loss_value}")
            per_seg.append(loss_value)
            group_loss = group_loss + l / N_SEGS
        if train:
            scaler.scale(group_loss).backward()
        model.detach_state()
    return per_seg


@torch.no_grad()
def evaluate(model, docs):
    acc = {"mem": [], "reset": [], "base": []}
    for i in range(0, len(docs), BATCH_DOCS):
        ids = docs[i:i + BATCH_DOCS].to(device)
        model.enabled = True
        acc["mem"].append(doc_losses(model, ids))
        acc["reset"].append(doc_losses(model, ids, reset_each=True))
        model.enabled = False
        acc["base"].append(doc_losses(model, ids))
        model.enabled = True
    res = {k: np.mean(v, axis=0) for k, v in acc.items()}      # theo segment
    summ = {k: float(v[1:].mean()) for k, v in res.items()}     # bỏ segment đầu (chưa có quá khứ)
    return summ, res

In [ ]:
CFG = dict(kind=MEMORY_KIND, model_name=MODEL_NAME, seg_len=SEG_LEN, n_segs=N_SEGS, mem_chunk=MEM_CHUNK, mem_heads=MEM_HEADS,
           mem_dim_head=MEM_DIM_HEAD, mem_depth=MEM_DEPTH, use_momentum=USE_MOMENTUM,
           titans_inner_lr_cap=TITANS_INNER_LR_CAP, titans_kwargs=TITANS_KWARGS,
           objective="fact_mix", fact_answer_weight=FACT_ANSWER_WEIGHT, layer=model.layer)

params = model.trainable_parameters()
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0, betas=(0.9, 0.95))
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

def save_ckpt(name, step, best, with_opt=True):
    os.makedirs(CKPT_DIR, exist_ok=True)
    blob = {"mem": model.mem.state_dict(), "out": model.out.state_dict(), "gate": model.gate.data.cpu(),
            "step": step, "best": best, "cfg": CFG, "factmix": FACT_SPLIT_META}
    if with_opt:
        blob["opt"] = opt.state_dict(); blob["scaler"] = scaler.state_dict()
    path = os.path.join(CKPT_DIR, name)
    torch.save(blob, path + ".tmp")
    os.replace(path + ".tmp", path)

def load_ckpt(path, with_opt=True):
    blob = torch.load(path, map_location=device)
    model.mem.load_state_dict(blob["mem"]); model.out.load_state_dict(blob["out"])
    model.gate.data = blob["gate"].to(device)
    if with_opt and "opt" in blob:
        opt.load_state_dict(blob["opt"]); scaler.load_state_dict(blob["scaler"])
    return blob["step"], blob["best"]

start_step, best_val = 0, float("inf")
if INIT_CKPT:
    if not os.path.isfile(INIT_CKPT):
        raise FileNotFoundError(f"INIT_CKPT được đặt nhưng không tồn tại: {INIT_CKPT}")
    resume_blob = torch.load(INIT_CKPT, map_location="cpu")
    resume_cfg = resume_blob.get("cfg", {})
    saved_cap = resume_cfg.get("titans_inner_lr_cap")
    if saved_cap is None or abs(float(saved_cap) - TITANS_INNER_LR_CAP) > 1e-12:
        raise ValueError(f"Không resume: checkpoint lr cap={saved_cap}, cấu hình hiện tại={TITANS_INNER_LR_CAP}")
    start_step, best_val = load_ckpt(INIT_CKPT)
    print("Đã resume từ", INIT_CKPT, "| bước", start_step, "| best", best_val)
else:
    print("Train v2 từ đầu; không resume checkpoint v1.")

## 4. Pretrain phần memory (outer loop)

Backbone không đổi. Loss là cross-entropy dự đoán token tiếp theo trên từng segment; để giảm loss ở segment sau, memory buộc phải ghi và truy xuất thông tin từ các segment trước. Checkpoint `last.pt` lưu định kỳ (có optimizer để resume), `best.pt` lưu theo loss đánh giá thấp nhất.

In [ ]:
def lr_at(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    p = (step - WARMUP) / max(1, MAX_STEPS - WARMUP)
    return LR * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, p))))

t0 = time.time()
model.mem.train(); model.out.train()
log_loss, step = [], start_step
order = torch.randperm(len(train_docs), generator=torch.Generator().manual_seed(SEED))   # cùng thứ tự dữ liệu ở mọi notebook
while step < MAX_STEPS:
    if (time.time() - t0) / 60 > MAX_MINUTES:
        print("Hết thời gian cho phép, dừng và lưu checkpoint.")
        break
    idx = order[(step * BATCH_DOCS) % len(train_docs):][:BATCH_DOCS]
    if len(idx) < BATCH_DOCS:
        idx = torch.randint(0, len(train_docs), (BATCH_DOCS,))
    ids = train_docs[idx].to(device)
    ids, token_weights = inject_factmix(ids, step)

    for g in opt.param_groups:
        g["lr"] = lr_at(step)
    opt.zero_grad(set_to_none=True)
    per_seg = doc_losses(model, ids, train=True, scaler=scaler, token_weights=token_weights)
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    scaler.step(opt); scaler.update()
    step += 1
    log_loss.append(float(np.mean(per_seg[1:])))

    if step % 10 == 0 or step == 1:
        print(f"bước {step}/{MAX_STEPS} | loss (seg>=1) {np.mean(log_loss[-10:]):.4f} | {(time.time()-t0)/60:.1f} phút")
    if step % EVAL_EVERY == 0 or step == MAX_STEPS:
        model.mem.eval(); model.out.eval()
        summ, _ = evaluate(model, eval_docs)
        print(f"  [eval] mem {summ['mem']:.4f} | reset {summ['reset']:.4f} | backbone thuần {summ['base']:.4f}"
              f" | lợi ích của quá khứ qua memory (reset-mem): {summ['reset']-summ['mem']:+.4f}")
        if summ["mem"] < best_val:
            best_val = summ["mem"]; save_ckpt("best.pt", step, best_val, with_opt=False)
        model.mem.train(); model.out.train()
    if step % SAVE_EVERY == 0:
        save_ckpt("last.pt", step, best_val)

save_ckpt("last.pt", step, best_val)
save_ckpt(f"{MEMORY_KIND}_vi_memory.pt", step, best_val, with_opt=False)
model.mem.eval(); model.out.eval()
final_blob = torch.load(os.path.join(CKPT_DIR, f"{MEMORY_KIND}_vi_memory.pt"), map_location="cpu")
torch.save(final_blob, "/kaggle/working/titans_v2.pt")
print("Đã lưu checkpoint /kaggle/working/titans_v2.pt | bước", step)
print("Titans init:", TITANS_KWARGS, "| outer gate:", float(model.gate.detach().cpu()))

## 5. Đánh giá: memory có giúp không?

Ba cấu hình trên cùng tập đánh giá chưa từng train:
- **mem**: memory mang trạng thái qua các segment.
- **reset**: cùng model nhưng xóa memory ở mỗi segment, tức không có quá khứ. Hiệu `reset − mem` là phần lợi ích thực sự do thông tin quá khứ.
- **base**: Qwen thuần, chỉ nhìn một segment (Transformer cùng cửa sổ).

In [ ]:
import matplotlib
if DEBUG: matplotlib.use("Agg")
import matplotlib.pyplot as plt

def checksum(module):
    return float(sum(p.detach().double().abs().sum().item() for p in module.parameters()))

@torch.no_grad()
def trace_run(model, ids):
    ids = ids.to(device)
    before = (checksum(model.lm), checksum(model.mem) + checksum(model.out))
    model.reset(); model.enabled = True; model.trace = True; model.calls, model.w_ref = [], None
    for s in range(0, len(ids), SEG_LEN):
        seg = ids[s:s + SEG_LEN][None]
        model.lm(input_ids=seg, use_cache=False)
    model.trace = False
    after = (checksum(model.lm), checksum(model.mem) + checksum(model.out))
    cat = lambda key, sl: np.concatenate([c[key][sl] for c in model.calls])
    return dict(dist=cat("dist", slice(1, None)), dist_abs=cat("dist_abs", slice(1, None)), step=cat("step", slice(None)),
                surprise=cat("surprise", slice(None)), theta=cat("theta", slice(None)),
                before=before, after=after, fast_params=model.fast_params(), n_tokens=len(ids), kind=model.kind)

def plot_trace(tr):
    cfg = lm.config
    n = tr["n_tokens"]
    xs_chunk = (np.arange(len(tr["dist"])) + 1) * MEM_CHUNK
    fig, ax = plt.subplots(2, 2, figsize=(11, 7))
    ax[0, 0].plot(xs_chunk, tr["dist"], label=f"{tr['kind']}: fast-weight của memory")
    ax[0, 0].plot(xs_chunk, np.zeros_like(xs_chunk, dtype=float), "--", label="Transformer thường: trọng số cố định")
    ax[0, 0].set_title("‖M_t − M_0‖ / ‖M_0‖ theo số token"); ax[0, 0].set_xlabel("token"); ax[0, 0].legend()
    k = max(1, MEM_CHUNK)
    sm = np.convolve(tr["surprise"], np.ones(k) / k, mode="valid")
    ax[0, 1].plot(np.arange(len(sm)) + k, sm, label="surprise (loss memory, làm mượt)")
    ax[0, 1].set_xlabel("token"); ax[0, 1].set_ylabel("surprise")
    a2 = ax[0, 1].twinx(); a2.plot(xs_chunk[:len(tr["step"])], tr["step"], color="tab:red", alpha=.6, label="‖ΔM‖ mỗi chunk")
    ax[0, 1].set_title("Surprise và độ lớn cập nhật"); ax[0, 1].legend(loc="upper left"); a2.legend(loc="upper right")
    ax[1, 0].plot(tr["theta"]); ax[1, 0].set_title("learning rate trong inner loop (θ_t / η_t)"); ax[1, 0].set_xlabel("token")
    head_dim = cfg.hidden_size // cfg.num_attention_heads
    toks = np.linspace(0, max(n, 4096), 50)
    kv_mb = 2 * cfg.num_hidden_layers * cfg.num_key_value_heads * head_dim * toks * 2 / 1e6   # fp16
    mem_mb = np.full_like(toks, tr["fast_params"] * 4 / 1e6)
    ax[1, 1].plot(toks, kv_mb, label="Transformer: KV cache (tăng theo độ dài)")
    ax[1, 1].plot(toks, mem_mb, label=f"{tr['kind']}: fast-weight memory (cố định)")
    ax[1, 1].set_title("Kích thước bộ nhớ (MB)"); ax[1, 1].set_xlabel("token"); ax[1, 1].legend()
    plt.tight_layout(); plt.show()
    print(f"Checksum backbone  trước: {tr['before'][0]:.6f} | sau: {tr['after'][0]:.6f} | đổi: {tr['after'][0] != tr['before'][0]}")
    print(f"Checksum outer mem trước: {tr['before'][1]:.6f} | sau: {tr['after'][1]:.6f} | đổi: {tr['after'][1] != tr['before'][1]}")
    print(f"Fast-weight đã dịch chuyển tới {tr['dist'][-1]:.4f} lần chuẩn ban đầu (0 = không đổi).")


summ, per_seg = evaluate(model, eval_docs)
print({k: round(v, 4) for k, v in summ.items()}, "(trung bình trên các segment từ thứ 2)")
fig, ax = plt.subplots(figsize=(7, 3.5))
for k, lab in [("base", "Qwen thuần (cửa sổ 1 segment)"), ("reset", "Có adapter, xóa memory mỗi segment"), ("mem", f"Memory ({MEMORY_KIND}) mang qua segment")]:
    ax.plot(range(1, N_SEGS + 1), per_seg[k], marker="o", label=lab)
ax.set_xlabel("Segment"); ax.set_ylabel("Loss"); ax.legend(); ax.set_title("Loss theo segment trên tập đánh giá")
plt.tight_layout(); plt.show()

## 6. Lúc test: trọng số thay đổi thế nào khi token đi vào

Mọi tham số đã train được **đóng băng**. `trace_run` cho văn bản đi qua từng segment và ghi lại: khoảng cách của fast-weight so với lúc bắt đầu `‖M_t − M_0‖ / ‖M_0‖`, độ lớn mỗi bước cập nhật, *surprise* (loss của memory trên từng token), learning rate, và checksum tham số backbone/outer trước-sau để chứng minh chỉ fast-weight đổi.

Một Transformer thường có `‖W_t − W_0‖ = 0` trong suốt quá trình (đường phẳng); bộ nhớ của nó là KV cache và tăng tuyến tính theo độ dài.

In [ ]:
demo_ids = eval_docs[0][: SEG_LEN * N_SEGS]
tr = trace_run(model, demo_ids)
plot_trace(tr)

## 7. Token nào được hưởng lợi từ memory

Tô màu từng token theo `loss(không có quá khứ) − loss(có memory)`. Xanh = memory giúp dự đoán tốt hơn, đỏ = tệ hơn. Chỉ có ý nghĩa từ segment thứ hai trở đi.

In [ ]:
@torch.no_grad()
def token_losses(model, ids, enabled=True):
    ids = ids.to(device)
    model.reset(); model.enabled = enabled
    losses, tids = [], []
    for s in range(0, len(ids), SEG_LEN):
        seg = ids[s:s + SEG_LEN][None]
        if seg.size(1) < 2:
            break
        logits = model.lm(input_ids=seg, use_cache=False).logits
        l = F.cross_entropy(logits[0, :-1].float(), seg[0, 1:], reduction="none")
        losses.append(l.cpu().numpy()); tids.append(seg[0, 1:].cpu().numpy())
    model.enabled = True
    return np.concatenate(losses), np.concatenate(tids)

def gain_html(model, ids, start_seg=1, max_tokens=400):
    l_mem, tids = token_losses(model, ids, True)
    l_base, _ = token_losses(model, ids, False)
    gain = (l_base - l_mem)
    lo = start_seg * (SEG_LEN - 1)
    gain, tids = gain[lo:lo + max_tokens], tids[lo:lo + max_tokens]
    mx = max(1e-6, float(np.percentile(np.abs(gain), 95)))
    parts = []
    for t, g in zip(tids, gain):
        a = min(1.0, abs(float(g)) / mx)
        col = f"rgba(0,160,0,{a:.2f})" if g > 0 else f"rgba(220,0,0,{a:.2f})"
        parts.append(f'<span style="background:{col}">{_html.escape(decode([int(t)]))}</span>')
    msg = (f"loss trung bình (đoạn hiển thị): backbone thuần {l_base[lo:lo+max_tokens].mean():.3f} | "
           f"có memory {l_mem[lo:lo+max_tokens].mean():.3f}")
    return "".join(parts), msg

def show_gain(model, ids, **kw):
    h, msg = gain_html(model, ids, **kw)
    display(HTML(h)); print(msg)

show_gain(model, demo_ids)

## 8. Kiểm chứng bằng sự kiện tự bịa (đáp án do bạn biết trước)

Tạo một "định luật" chưa tồn tại trong dữ liệu nào, chèn vào đầu, rồi thêm nhiều văn bản nhiễu dài hơn cửa sổ, cuối cùng hỏi lại. So ba cách: (1) **cửa sổ thường** (backbone chỉ thấy câu hỏi, cận dưới), (2) **memory**, (3) **toàn bộ ngữ cảnh trong cửa sổ** (cận trên, chỉ khả thi khi vừa context). Đáp án chuẩn do bạn gõ nên đối chiếu trực tiếp được. Nếu memory chưa đủ mạnh, kết quả sẽ không tốt hơn cửa sổ thường: hãy báo đúng như vậy.

In [ ]:
@torch.no_grad()
def answer_score(model, context_ids, prompt_ids, answer_ids, mode):
    ans = torch.tensor(answer_ids, device=device)
    pr = torch.tensor(prompt_ids, device=device)
    if mode == "memory":
        model.reset(); model.enabled = True
        for s in range(0, len(context_ids), SEG_LEN):
            model.lm(input_ids=context_ids[s:s + SEG_LEN][None].to(device), use_cache=False)
        seq = torch.cat([pr, ans])
    elif mode == "window":
        model.enabled = False; model.reset()
        seq = torch.cat([pr, ans])
    else:
        model.enabled = False; model.reset()
        seq = torch.cat([context_ids.to(device), pr, ans])
    n = len(answer_ids)
    logits = model.lm(input_ids=seq[None], use_cache=False, logits_to_keep=n + 1).logits[0].float()
    model.enabled = True
    logp = F.log_softmax(logits[-n - 1:-1], dim=-1)
    tok_lp = logp[torch.arange(n), ans]
    top1 = (logp.argmax(-1) == ans).all().item()
    return float(tok_lp.sum()), float(tok_lp.exp().prod()), bool(top1)


## 9. Hỏi trên tài liệu của bạn (sách / chương vật lý)

Dán văn bản vào `DOC_TEXT`, đặt `PROMPT2`/`ANSWER2` là câu điền khuyết lấy từ chính tài liệu. Chọn chi tiết mà Qwen không thể biết trước (số liệu riêng, quy ước riêng), nếu không tín hiệu của memory sẽ lẫn với kiến thức có sẵn của Qwen.

In [ ]:
DOC_TEXT = ""      # dán một chương sách (khoảng 50-100KB) vào đây
PROMPT2  = ""      # ví dụ: "Theo giáo trình, đơn vị của đại lượng X là"
ANSWER2  = ""      # ví dụ: " kilogam"

if DOC_TEXT and PROMPT2 and ANSWER2:
    ctx = torch.tensor(encode(DOC_TEXT))
    p2, a2 = encode(PROMPT2), encode(ANSWER2)
    print("Độ dài tài liệu (token):", len(ctx))
    tr2 = trace_run(model, ctx)
    plot_trace(tr2)
    for mode, name in [("window", "Cửa sổ thường"), ("memory", f"Memory ({MEMORY_KIND})"), ("full", "Toàn bộ ngữ cảnh")]:
        try:
            lp, pr, ok = answer_score(model, ctx, p2, a2, mode)
            print(f"{name:20s} | log P(đáp án) = {lp:8.3f} | P = {pr:.4g} | top-1 đúng: {ok}")
        except torch.cuda.OutOfMemoryError:
            print(f"{name:20s} | hết bộ nhớ GPU (tài liệu quá dài cho chế độ này)")
else:
    print("Chưa nhập DOC_TEXT/PROMPT2/ANSWER2, bỏ qua mục này.")

## 10. Xuất file cho webapp và notebook so sánh

File `titans_vi_memory.pt` (không kèm optimizer) chứa trọng số `mem`, `out`, `gate` và cấu hình. Tải từ tab *Output* của Kaggle.

In [ ]:
final = os.path.join(CKPT_DIR, f"{MEMORY_KIND}_vi_memory.pt")
print(final, "| kích thước MB:", round(os.path.getsize(final) / 1e6, 1))

## 11. Held-out retrieval evaluation (Titans only)
Validation/test facts and wording are disjoint from training; no TTT comparison is run in this notebook.


In [ ]:
@torch.no_grad()
def rank_titans_candidates(context_ids, prompt_ids, values):
    rows=[]
    for value in values:
        ans_ids=encode(" "+value)
        lp, _, _ = answer_score(model, context_ids, prompt_ids, ans_ids, "memory")
        rows.append({"value":value,"logp":lp})
    rows.sort(key=lambda x:-x["logp"])
    for i,row in enumerate(rows): row["rank"]=i+1
    return rows

def retrieval_context(fact, fact_index, distance):
    ids=eval_docs[fact_index % len(eval_docs)].clone()
    event_text=(FACT_SPLIT_META["eval_template"].format(**fact))
    event_ids=torch.tensor(encode(event_text),dtype=ids.dtype)
    event_segment=ids[:SEG_LEN].clone()
    event_segment[-len(event_ids):]=event_ids
    noise=ids[SEG_LEN:(distance+1)*SEG_LEN]
    return torch.cat([event_segment,noise])

def score_fact_set(facts, distances, eval_seeds):
    rows=[]
    for distance in distances:
        for fact_i,fact in enumerate(facts):
            prompt=f"Truy xu?t phi?u l?u tr? t?i tr?m {fact['name']}: ch? s? h?p kim {fact['metal']} l?"
            prompt_ids=encode(prompt)
            for eval_seed in eval_seeds:
                rng=random.Random(eval_seed*100003+fact_i*97+distance)
                candidates=[fact["value"]]
                while len(candidates)<4:
                    value=f"{rng.randint(1,9)},{rng.randint(10,99):02d}"
                    if value not in candidates: candidates.append(value)
                context=retrieval_context(fact,fact_i,distance)
                ranking=rank_titans_candidates(context,prompt_ids,candidates)
                gold_rank=next(r["rank"] for r in ranking if r["value"]==fact["value"])
                gold_logp=next(r["logp"] for r in ranking if r["value"]==fact["value"])
                rows.append({"distance":distance,"fact_index":fact_i,"seed":eval_seed,
                             "gold_rank":gold_rank,"top1":float(gold_rank==1),
                             "mrr":1.0/gold_rank,"gold_logp":gold_logp})
    return rows

def bootstrap_ci(values, seed, rounds=2000):
    rng=np.random.default_rng(seed); a=np.asarray(values,dtype=float)
    samples=rng.choice(a,size=(rounds,len(a)),replace=True).mean(axis=1)
    return [float(np.quantile(samples,.025)),float(np.quantile(samples,.975))]

# Separate, fixed validation and test facts; only Titans is evaluated in this run.
VAL_SEEDS=[17,29]
val_rows=score_fact_set(VAL_FACTS,[2,4],VAL_SEEDS)
test_rows=score_fact_set(TEST_FACTS,FACTMIX_DISTANCES,[17,29])
val_summary={str(d):{"top1":float(np.mean([r["top1"] for r in val_rows if r["distance"]==d])),
                     "mrr":float(np.mean([r["mrr"] for r in val_rows if r["distance"]==d]))}
             for d in [2,4]}
test_summary={}
for distance in FACTMIX_DISTANCES:
    group=[r for r in test_rows if r["distance"]==distance]
    test_summary[str(distance)]={"top1":float(np.mean([r["top1"] for r in group])),
        "top1_ci95":bootstrap_ci([r["top1"] for r in group],500+distance),
        "mrr":float(np.mean([r["mrr"] for r in group])),
        "mean_gold_logp":float(np.mean([r["gold_logp"] for r in group])),
        "n":len(group)}
    print(f"test distance={distance} | top1={test_summary[str(distance)]['top1']:.3f} "
          f"| MRR={test_summary[str(distance)]['mrr']:.3f} "
          f"| gold logP={test_summary[str(distance)]['mean_gold_logp']:.3f} "
          f"| CI95={test_summary[str(distance)]['top1_ci95']}")

# Log the trained Titans alpha / eta / momentum values on a fixed held-out segment.
captured={}
def capture_hidden(module,args,output): captured["h"]=output[0] if isinstance(output,tuple) else output
handle=lm.model.layers[model.layer].register_forward_hook(capture_hidden)
model.enabled=False; lm(input_ids=eval_docs[:1,:SEG_LEN].to(device),use_cache=False)
handle.remove(); h=captured["h"].float()
chunk_h=h[:,:SEG_LEN//MEM_CHUNK*MEM_CHUNK].reshape(1,-1,MEM_CHUNK,h.size(-1)).mean(2)
alpha=float(model.mem.to_decay_factor(chunk_h).sigmoid().mean().cpu())
eta=float(model.mem.adaptive_step_transform(model.mem.to_adaptive_step(h)).mean().cpu())
momentum=float(model.mem.to_momentum(chunk_h).sigmoid().mean().cpu())
print(f"trained mean alpha={alpha:.6f} | eta={eta:.6f} | momentum={momentum:.6f} | outer gate={float(model.gate.detach().cpu()):.6f}")
model.enabled=True

# Same broad schema as the app's Kaggle reference, with Titans v2 results only.
summary_loss,_=evaluate(model,eval_docs)
eval_out={
  "source":"Titans v2 fact-mix run on Kaggle GPU; single training seed; held-out synthetic retrieval facts",
  "backbone":MODEL_NAME,
  "setup":{"seg_len":SEG_LEN,"n_segs":N_SEGS,"mem_chunk":MEM_CHUNK,"heads":MEM_HEADS,
           "train_steps":step,"train_seed":SEED,"fact_answer_weight":FACT_ANSWER_WEIGHT,
           "titans_kwargs":TITANS_KWARGS,"titans_inner_lr_cap":TITANS_INNER_LR_CAP,
           "alpha_mean":alpha,"eta_mean":eta,"momentum_mean":momentum,"outer_gate":float(model.gate.detach().cpu())},
  "validation":{"facts":VAL_FACTS,"seeds":VAL_SEEDS,"by_distance":val_summary},
  "test":{"facts":TEST_FACTS,"seeds":VAL_SEEDS,"by_distance":test_summary},
  "eval_loss":{"titans":{"memory":summary_loss["mem"],"reset":summary_loss["reset"],"base":summary_loss["base"],
              "benefit_reset_minus_memory":summary_loss["reset"]-summary_loss["mem"]}},
  "invented_fact_recall":{"distances_segments":FACTMIX_DISTANCES,"n_facts_per_distance":len(TEST_FACTS),
              "titans":{"logp":[test_summary[str(d)]["mean_gold_logp"] for d in FACTMIX_DISTANCES],
                         "top1":[test_summary[str(d)]["top1"] for d in FACTMIX_DISTANCES],
                         "mrr":[test_summary[str(d)]["mrr"] for d in FACTMIX_DISTANCES],
                         "top1_ci95":[test_summary[str(d)]["top1_ci95"] for d in FACTMIX_DISTANCES]}}
}
with open("/kaggle/working/eval_v2.json","w",encoding="utf-8") as f: json.dump(eval_out,f,ensure_ascii=False,indent=2)
with open("/kaggle/working/titans_v2.log","w",encoding="utf-8") as f: json.dump({"eval":eval_out,"train_loss_log":log_loss},f,ensure_ascii=False,indent=2)
print("Saved /kaggle/working/eval_v2.json and titans_v2.log")
